In [0]:
%sql
DESCRIBE TABLE _exponent.omop.visit_detail

In [0]:
%sql
SELECT * FROM _exponent._bronze_allscripts_tw_works_vw.dbo_encounter_type_de

In [0]:
%sql
SELECT
source_to_person.person_id AS person_id, --required
-- visit_detail_concept_id -- required
-- visit_detail_start_date -- required
-- visit_detail_start_datetime
-- visit_detail_end_date -- required
-- visit_detail_end_datetime
-- visit_detail_type_concept_id -- required
-- provider_id
-- care_site_id
CONCAT_WS(
       chr(31),
       'allscripts_tw',
       'dbo_encounter',
       'id',
       CAST(dbo_encounter.id AS BIGINT)
     ) AS visit_detail_source_value,
-- visit_detail_source_concept_id
-- admitted_from_concept_id
-- admitted_from_source_value
-- discharged_to_concept_id
-- discharged_to_source_value
-- preceding_visit_detail_id
-- parent_visit_detail_id
source_to_visit_occurrence.visit_occurrence_id AS visit_occurrence_id -- required

FROM _exponent._bronze_allscripts_tw_works_vw.dbo_encounter
LEFT JOIN _exponent._bronze_allscripts_tw_works_vw.dbo_encounter_type_de
ON dbo_encounter_type_de.id = dbo_encounter.encountertypede
LEFT JOIN _exponent._bronze_allscripts_tw_works_vw.dbo_person
ON dbo_person.id = dbo_encounter.patientid
LEFT JOIN _exponent._bronze_allscripts_tw_works_vw.dbo_visit
ON dbo_visit.id = dbo_encounter.visitid
LEFT JOIN _exponent.omop_mapping.source_to_person
  ON source_to_person.person_source_value = CONCAT_WS(
       chr(31),
       'allscripts_tw',
       'dbo_person',
       'id',
       CAST(dbo_person.id AS BIGINT)
     )
LEFT JOIN _exponent.omop_mapping.source_to_visit_occurrence
  ON source_to_visit_occurrence.visit_occurrence_source_value = CONCAT_WS(
       chr(31),
       'allscripts_tw',
       'dbo_visit',
       'id',
       CAST(dbo_visit.id AS BIGINT)
     )

LIMIT 50

In [0]:
%sql
WITH encounter_dates_cte AS (
  SELECT
    dbo_encounter.id AS encounter_id,
    CASE
      WHEN dbo_encounter.dttm IS NULL THEN NULL
      WHEN CAST(dbo_encounter.dttm AS STRING) = '-' THEN NULL
      WHEN CAST(dbo_encounter.dttm AS TIMESTAMP) < TIMESTAMP('1970-01-01') THEN NULL
      ELSE CAST(dbo_encounter.dttm AS TIMESTAMP)
    END AS clean_dttm
  FROM _exponent._bronze_allscripts_tw_works_vw.dbo_encounter
)

SELECT
  source_to_person.person_id AS person_id, -- required
  COALESCE(visit_detail_concept.omop_concept_id, 0) AS visit_detail_concept_id, -- required
  CAST(encounter_dates_cte.clean_dttm AS DATE) AS visit_detail_start_date, -- required
  encounter_dates_cte.clean_dttm AS visit_detail_start_datetime,
  CAST(encounter_dates_cte.clean_dttm AS DATE) AS visit_detail_end_date, -- required
  encounter_dates_cte.clean_dttm AS visit_detail_end_datetime,
  CAST(32817 AS INT) AS visit_detail_type_concept_id, -- required -- 32817 = EHR
  source_to_provider.provider_id AS provider_id,
  source_to_care_site.care_site_id AS care_site_id,
  CONCAT_WS(
    chr(31),
    'allscripts_tw',
    'dbo_encounter',
    'id',
    CAST(dbo_encounter.id AS BIGINT)
  ) AS visit_detail_source_value,
  CAST(NULL AS INT) AS visit_detail_source_concept_id,
  CAST(NULL AS INT) AS admitted_from_concept_id,
  CAST(NULL AS STRING) AS admitted_from_source_value,
  CAST(NULL AS INT) AS discharged_to_concept_id,
  CAST(NULL AS STRING) AS discharged_to_source_value,
  CAST(NULL AS BIGINT) AS preceding_visit_detail_id,
  CAST(NULL AS BIGINT) AS parent_visit_detail_id,
  source_to_visit_occurrence.visit_occurrence_id AS visit_occurrence_id, -- required
  'allscripts_tw' AS source_system

FROM _exponent._bronze_allscripts_tw_works_vw.dbo_encounter

LEFT JOIN _exponent._bronze_allscripts_tw_works_vw.dbo_encounter_type_de
  ON dbo_encounter_type_de.id = dbo_encounter.encountertypede
 AND UPPER(dbo_encounter_type_de.isinactiveflag) = 'N'

LEFT JOIN _exponent._bronze_allscripts_tw_works_vw.dbo_person
  ON dbo_person.id = dbo_encounter.patientid

LEFT JOIN _exponent._bronze_allscripts_tw_works_vw.dbo_visit
  ON dbo_visit.id = dbo_encounter.visitid

LEFT JOIN encounter_dates_cte
  ON encounter_dates_cte.encounter_id = dbo_encounter.id

LEFT JOIN _exponent.omop_mapping.source_to_person
  ON source_to_person.person_source_value = CONCAT_WS(
    chr(31),
    'allscripts_tw',
    'dbo_person',
    'id',
    CAST(dbo_person.id AS BIGINT)
  )
 AND source_to_person.active_flag = TRUE

LEFT JOIN _exponent.omop_mapping.source_to_visit_occurrence
  ON source_to_visit_occurrence.visit_occurrence_source_value = CONCAT_WS(
    chr(31),
    'allscripts_tw',
    'dbo_visit',
    'id',
    CAST(dbo_visit.id AS BIGINT)
  )
 AND source_to_visit_occurrence.active_flag = TRUE

-- Provider mapping (uses dbo_encounter.providerid)
LEFT JOIN _exponent.omop_mapping.source_to_provider
  ON source_to_provider.provider_source_value = CONCAT_WS(
    chr(31),
    'allscripts_tw',
    'dbo_provider',
    'id',
    CAST(dbo_encounter.providerid AS BIGINT)
  )
 AND source_to_provider.active_flag = TRUE

-- Care site mapping (prefer DefaultBillingLocationDE, fallback to PrimaryLocationDE)
LEFT JOIN _exponent.omop_mapping.source_to_care_site
  ON source_to_care_site.care_site_source_value = CONCAT_WS(
    chr(31),
    'allscripts_tw',
    'dbo_billing_location_de',
    'id',
    CAST(
      COALESCE(
        NULLIF(CAST(dbo_encounter.defaultbillinglocationde AS BIGINT), 0),
        NULLIF(CAST(dbo_encounter.primarylocationde AS BIGINT), 0)
      ) AS BIGINT
    )
  )
 AND source_to_care_site.active_flag = TRUE

-- Map Encounter Type -> OMOP Visit concept for visit_detail_concept_id
LEFT JOIN _exponent.omop_mapping.domain_source_to_concept visit_detail_concept
  ON visit_detail_concept.source_system = 'allscripts_tw'
 AND visit_detail_concept.source_table  = 'dbo_encounter_type_de'
 AND LOWER(visit_detail_concept.source_field) = 'entryname'
 AND visit_detail_concept.domain_id     = 'Visit'
 AND visit_detail_concept.source_value  = dbo_encounter_type_de.entryname
 AND visit_detail_concept.active_flag   = TRUE

WHERE dbo_encounter.id IS NOT NULL
  AND encounter_dates_cte.clean_dttm IS NOT NULL

LIMIT 50;

In [0]:

%sql
SELECT *
FROM _exponent._bronze_allscripts_tw_works_vw.dbo_encounter
LIMIT 1000

In [0]:
%sql
SELECT
-- visit_detail_id --required
source_to_person.person_id AS person_id --required
-- visit_detail_concept_id --required
-- visit_detail_start_date --required
-- visit_detail_start_datetime
-- visit_detail_end_date --required
-- visit_detail_end_datetime
-- visit_detail_type_concept_id --required
-- provider_id
-- care_site_id
-- visit_detail_source_value
-- visit_detail_source_concept_id
-- admitted_from_concept_id
-- admitted_from_source_value
-- discharged_to_concept_id
-- discharged_to_source_value
-- preceding_visit_detail_id
-- parent_visit_detail_id
-- visit_occurrence_id --required
FROM _exponent._bronze_allscripts_tw_works_vw.dbo_visit
LEFT JOIN _exponent._bronze_allscripts_tw_works_vw.dbo_visit_detail
ON dbo_visit_detail.visitid = dbo_visit.id 
LEFT JOIN _exponent._bronze_allscripts_tw_works_vw.dbo_visit_type_de
ON dbo_visit_type_de.id = dbo_visit.visittypede 
LEFT JOIN _exponent._bronze_allscripts_tw_works_vw.dbo_visit_status_de
ON dbo_visit_status_de.id = dbo_visit.visitstatusde
LEFT JOIN _exponent._bronze_allscripts_tw_works_vw.dbo_person AS dbo_person
  ON dbo_person.id = dbo_visit.patientid
LEFT JOIN _exponent._bronze_allscripts_tw_works_vw.dbo_provider AS dbo_provider
  ON dbo_provider.id = COALESCE(
       dbo_visit_detail.attendingproviderid,
       dbo_visit_detail.admittingproviderid,
       dbo_visit.primaryproviderid
     )
LEFT JOIN _exponent.omop_mapping.source_to_person
  ON source_to_person.person_source_value = CONCAT_WS(
       CHR(31),
       'allscripts_tw',
       'dbo_person',
       'id',
       CAST(dbo_person.id AS BIGINT)
     )
 AND source_to_person.active_flag = TRUE
LEFT JOIN _exponent.omop_mapping.domain_source_to_concept AS visit_concept
  ON visit_concept.source_id = dbo_visit_type_de.id
 AND visit_concept.source_value = dbo_visit_type_de.entryname
 AND visit_concept.domain_id = 'Visit'
 AND visit_concept.source_system = 'allscripts_tw'
 AND visit_concept.source_table = 'dbo_visit_type_de'
 AND visit_concept.source_field = 'entryname'
 AND visit_concept.active_flag = TRUE
 LEFT JOIN _exponent.omop_mapping.source_to_provider AS source_to_provider
  ON source_to_provider.provider_source_value = CONCAT_WS(
       CHR(31),
       'allscripts_tw',
       'dbo_provider',
       'id',
       CAST(dbo_provider.id AS BIGINT)
     )
 AND source_to_provider.active_flag = TRUE


In [0]:
%sql
%sql
CREATE OR REPLACE TEMPORARY VIEW visit_occurrence_silver AS 
WITH final_discharge_per_visit AS (
  SELECT
    dbo_encounter.visitid AS visitid,
    dbo_encounter_dischargedisposition.dischargedispositionde,
    dbo_encounter_dischargedisposition.dischargedispositiondttm,
    dbo_encounter.dttm AS encounter_dttm,
    ROW_NUMBER() OVER (
      PARTITION BY dbo_encounter.visitid
      ORDER BY
        COALESCE(dbo_encounter_dischargedisposition.dischargedispositiondttm, dbo_encounter.dttm) DESC,
        dbo_encounter.id DESC
    ) AS rn
  FROM _exponent._bronze_allscripts_tw_works_vw.dbo_encounter
  LEFT JOIN _exponent._bronze_allscripts_tw_works_vw.dbo_encounter_dischargedisposition 
    ON dbo_encounter_dischargedisposition.encounterid = dbo_encounter.id
)

SELECT
  CONCAT_WS(
    CHR(31),
    'allscripts_tw',
    'dbo_visit',
    'id',
    CAST(dbo_visit.id AS BIGINT)
  ) AS visit_occurrence_source_value,
  source_to_person.person_id AS person_id,
  COALESCE(visit_concept.omop_concept_id, 9202) AS visit_concept_id,
  CAST(
    CASE
      WHEN dbo_visit_type_de.isinpatientflag = 'Y'
        THEN COALESCE(dbo_visit_detail.admissiondttm, dbo_visit.startdttm)
      ELSE dbo_visit.startdttm
    END
  AS DATE) AS visit_start_date,
  CASE
    WHEN dbo_visit_type_de.isinpatientflag = 'Y'
      THEN COALESCE(dbo_visit_detail.admissiondttm, dbo_visit.startdttm)
    ELSE dbo_visit.startdttm
  END AS visit_start_datetime,
  CAST(
    COALESCE(
      CASE
        WHEN dbo_visit_type_de.isinpatientflag = 'Y' THEN dbo_visit_detail.dischargedttm
      END,
      dbo_visit.enddttm,
      dbo_visit.startdttm
    )
  AS DATE) AS visit_end_date,
  COALESCE(
    CASE
      WHEN dbo_visit_type_de.isinpatientflag = 'Y' THEN dbo_visit_detail.dischargedttm
    END,
    dbo_visit.enddttm,
    dbo_visit.startdttm
  ) AS visit_end_datetime,
  CAST(32817 AS INT) AS visit_type_concept_id, -- 32817 = EHR
  source_to_provider.provider_id AS provider_id,

  NULL AS care_site_id, -- TODO: map locationde/billinglocationde once care_site is populated

  dbo_visit_type_de.entryname AS visit_source_value,
  CAST(0 AS INT) AS visit_source_concept_id,

  -- COALESCE(admitted_from_concept.omop_concept_id, 0) AS admitted_from_concept_id,
  NULL AS admitted_from_concept_id,

  -- dbo_arrival_mode_de.entryname AS admitted_from_source_value,
  NULL AS admitted_from_source_value,

  -- COALESCE(discharged_to_concept.omop_concept_id, 0) AS discharged_to_concept_id,
  NULL AS discharged_to_concept_id,
  dbo_discharge_disposition_de.entryname AS discharged_to_source_value,

  NULL AS preceding_visit_occurrence_id, -- TODO: sequencing logic

  'allscripts_tw' AS source_system

FROM _exponent._bronze_allscripts_tw_works_vw.dbo_visit AS dbo_visit

LEFT JOIN _exponent._bronze_allscripts_tw_works_vw.dbo_visit_detail
  ON dbo_visit_detail.visitid = dbo_visit.id

LEFT JOIN _exponent._bronze_allscripts_tw_works_vw.dbo_visit_type_de
  ON dbo_visit_type_de.id = dbo_visit.visittypede

-- LEFT JOIN _exponent._bronze_allscripts_tw_works_vw.dbo_visit_status_de
--   ON dbo_visit_status_de.id = dbo_visit.visitstatusde
LEFT JOIN _exponent._bronze_allscripts_tw_works_vw.dbo_person AS dbo_person
  ON dbo_person.id = dbo_visit.patientid
LEFT JOIN _exponent._bronze_allscripts_tw_works_vw.dbo_provider AS dbo_provider
  ON dbo_provider.id = COALESCE(
       dbo_visit_detail.attendingproviderid,
       dbo_visit_detail.admittingproviderid,
       dbo_visit.primaryproviderid
     )
-- LEFT JOIN _exponent._bronze_allscripts_tw_works_vw.dbo_arrival_mode_de
--   ON dbo_arrival_mode_de.id = dbo_visit.arrivalmodede
LEFT JOIN final_discharge_per_visit AS final_discharge_per_visit
  ON final_discharge_per_visit.visitid = dbo_visit.id
 AND final_discharge_per_visit.rn = 1

LEFT JOIN _exponent._bronze_allscripts_tw_works_vw.dbo_discharge_disposition_de
  ON dbo_discharge_disposition_de.id = final_discharge_per_visit.dischargedispositionde
LEFT JOIN _exponent.omop_mapping.source_to_person AS source_to_person
  ON source_to_person.person_source_value = CONCAT_WS(
       CHR(31),
       'allscripts_tw',
       'dbo_person',
       'id',
       CAST(dbo_person.id AS BIGINT)
     )
 AND source_to_person.active_flag = TRUE
LEFT JOIN _exponent.omop_mapping.source_to_provider AS source_to_provider
  ON source_to_provider.provider_source_value = CONCAT_WS(
       CHR(31),
       'allscripts_tw',
       'dbo_provider',
       'id',
       CAST(dbo_provider.id AS BIGINT)
     )
 AND source_to_provider.active_flag = TRUE

LEFT JOIN _exponent.omop_mapping.domain_source_to_concept AS visit_concept
  ON visit_concept.source_id = dbo_visit_type_de.id
 AND visit_concept.source_value = dbo_visit_type_de.entryname
 AND visit_concept.domain_id = 'Visit'
 AND visit_concept.source_system = 'allscripts_tw'
 AND visit_concept.source_table = 'dbo_visit_type_de'
 AND visit_concept.source_field = 'entryname'
 AND visit_concept.active_flag = TRUE
-- LEFT JOIN _exponent.omop_mapping.domain_source_to_concept AS admitted_from_concept
--   ON admitted_from_concept.source_id = dbo_arrival_mode_de.id
--  AND admitted_from_concept.source_value = dbo_arrival_mode_de.entryname
--  AND admitted_from_concept.domain_id = 'Visit'
--  AND admitted_from_concept.source_system = 'allscripts_tw'
--  AND admitted_from_concept.source_table = 'dbo_arrival_mode_de'
--  AND admitted_from_concept.source_field = 'entryname'
--  AND admitted_from_concept.active_flag = TRUE
-- LEFT JOIN _exponent.omop_mapping.domain_source_to_concept AS discharged_to_concept
--   ON discharged_to_concept.source_id = dbo_discharge_disposition_de.id
--  AND discharged_to_concept.source_value = dbo_discharge_disposition_de.entryname
--  AND discharged_to_concept.domain_id = 'Visit'
--  AND discharged_to_concept.source_system = 'allscripts_tw'
--  AND discharged_to_concept.source_table = 'dbo_discharge_disposition_de'
--  AND discharged_to_concept.source_field = 'entryname'
--  AND discharged_to_concept.active_flag = TRUE

WHERE dbo_visit.id IS NOT NULL
  AND dbo_visit.patientid IS NOT NULL
  -- AND (dbo_visit_status_de.isinactiveflag IS NULL OR dbo_visit_status_de.isinactiveflag = 'N')
;